# Ensemble: CNN + BiLSTM (No ESM) with K-Fold Cross-Validation

This notebook trains an ensemble model combining CNN and BiLSTM architectures with learned embeddings (no pre-trained embeddings) using K-fold cross-validation and early stopping.

In [ ]:
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from tqdm import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


## 1. Data

In [ ]:
seq_df = pd.read_csv('data/2018-06-06-pdb-intersect-pisces.csv', usecols=['pdb_id','seq'])
lab_df = pd.read_csv('data/2018-06-06-ss.cleaned.csv', usecols=['pdb_id','sst8','sst3'])
df = pd.merge(seq_df, lab_df, on='pdb_id', how='inner')
df['seq'] = df['seq'].str.replace('*','X')
df = df.dropna(subset=['seq','sst8','sst3']).copy()
df = df[(df['seq'].str.len()==df['sst8'].str.len()) & (df['seq'].str.len()==df['sst3'].str.len())].reset_index(drop=True)
df['len'] = df['seq'].str.len()
print(df.shape)
ss8_vocab = {'H':0,'G':1,'I':2,'E':3,'B':4,'T':5,'S':6,'C':7}
ss3_vocab = {'H':0,'E':1,'C':2}
chars = sorted(set(''.join(df['seq'])))
seq_vocab = {c:i+1 for i,c in enumerate(chars)}
seq_vocab['<pad>'] = 0
vocab_size = len(seq_vocab)
vocab_size


## 2. K-Fold Cross-Validation Setup - Dataset + Loaders
Prepare data for K-fold cross-validation with train/validation/test splits and custom collation.

In [ ]:
from sklearn.model_selection import KFold

class DS(Dataset):
    def __init__(self, seqs, s8, s3): self.seqs,self.s8,self.s3=seqs,s8,s3
    def __len__(self): return len(self.seqs)
    def __getitem__(self,i):
        t=[seq_vocab.get(c,0) for c in self.seqs[i]]
        a=[ss8_vocab.get(c,-1) for c in self.s8[i]]
        b=[ss3_vocab.get(c,-1) for c in self.s3[i]]
        return torch.tensor(t), torch.tensor(a), torch.tensor(b)

def collate(b):
    x,a,b = zip(*b)
    return (pad_sequence(x,batch_first=True,padding_value=seq_vocab['<pad>']),
            pad_sequence(a,batch_first=True,padding_value=-1),
            pad_sequence(b,batch_first=True,padding_value=-1))

# K-Fold Cross-Validation Setup: Split data into train+val (90%) and test (10%)
n = len(df)
n_test = int(n * 0.1)
test_indices = list(range(n - n_test, n))
train_val_indices = list(range(n - n_test))

# Test dataset
ds_te = DS(df.iloc[test_indices]['seq'].tolist(), df.iloc[test_indices]['sst8'].tolist(), df.iloc[test_indices]['sst3'].tolist())

# K-Fold configuration
n_folds = 5
kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)

opt = dict(num_workers=2, pin_memory=True)
test_loader = DataLoader(ds_te, batch_size=16, shuffle=False, collate_fn=collate, **opt)

embedding_dim = 128

print(f'K-Fold Setup:')
print(f'  Train+Val: {len(train_val_indices)} samples (for K-fold cross-validation)')
print(f'  Test: {len(ds_te)} samples (held-out)')
print(f'  Embedding dimension: {embedding_dim}')


## 3. Models

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__(); self.drop=nn.Dropout(dropout)
        import math
        pe = torch.zeros(max_len, d_model); pos=torch.arange(0,max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div); pe=pe.unsqueeze(0); self.register_buffer('pe', pe)
    def forward(self,x): x=x+self.pe[:,:x.size(1),:]; return self.drop(x)
class CNN(nn.Module):
    def __init__(self, vocab_size, input_dim=128, num_filters=128, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, input_dim, padding_idx=seq_vocab['<pad>'])
        self.pos = PositionalEncoding(input_dim, dropout)
        self.c1 = nn.Conv1d(input_dim, num_filters, 3, padding=1); self.r1=nn.ReLU(); self.d1=nn.Dropout(dropout)
        self.c2 = nn.Conv1d(num_filters, num_filters, 5, padding=2); self.r2=nn.ReLU(); self.d2=nn.Dropout(dropout)
        self.c3 = nn.Conv1d(num_filters, num_filters, 7, padding=3); self.r3=nn.ReLU(); self.d3=nn.Dropout(dropout)
        self.q8 = nn.Linear(num_filters,8); self.q3 = nn.Linear(num_filters,3)
    def forward(self,x):
        x=self.emb(x); x=self.pos(x); x=x.permute(0,2,1)
        x=self.d1(self.r1(self.c1(x))); x=self.d2(self.r2(self.c2(x))); x=self.d3(self.r3(self.c3(x)))
        x=x.permute(0,2,1); return self.q8(x), self.q3(x)
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden=256, dropout=0.3):
        super().__init__(); self.pad=seq_vocab['<pad>']
        self.emb=nn.Embedding(vocab_size, embed_dim, padding_idx=self.pad)
        self.l1=nn.LSTM(embed_dim, hidden, bidirectional=True, batch_first=True)
        self.l2=nn.LSTM(hidden*2, hidden, bidirectional=True, batch_first=True)
        self.drop=nn.Dropout(dropout); self.q8=nn.Linear(hidden*2,8); self.q3=nn.Linear(hidden*2,3)
    def forward(self,x):
        e=self.emb(x); L=(x!=self.pad).sum(1).to(torch.int64).cpu(); orig=e.size(1)
        p,_=self.l1(pack_padded_sequence(e,L,batch_first=True,enforce_sorted=False));
        p,_=self.l2(p); x,_=pad_packed_sequence(p,batch_first=True,total_length=orig)
        x=self.drop(x); return self.q8(x), self.q3(x)
cnn = CNN(vocab_size, input_dim=embedding_dim).to(device)
bilstm = BiLSTM(vocab_size, embed_dim=embedding_dim).to(device)
if torch.cuda.device_count()>1: cnn=nn.DataParallel(cnn); bilstm=nn.DataParallel(bilstm)
cnn, bilstm


## 4. Training with K-Fold Cross-Validation and Early Stopping
**Training Configuration:**
- **K-Fold Cross-Validation**: 5 folds
- **Epochs**: 50 epochs per fold
- **Early Stopping**: Patience of 5 epochs (stops if validation accuracy doesn't improve)
- **Models**: CNN and BiLSTM trained separately with learned embeddings
- **Embeddings**: Learned from scratch (no pre-trained embeddings)
- **Metrics**: Q8/Q3 accuracy and SOV (Segment Overlap) score
- **Ensemble**: Weighted combination of predictions tuned on validation set

In [ ]:
from torch.utils.data import Subset

def acc(logits, y): p=logits.argmax(-1); m=y!=-1; return (p[m]==y[m]).float().mean().item() if m.any() else 0.0

q3_id_to_char={0:'H',1:'E',2:'C'}
def get_segments(chars,state): seg=[]; st=-1
    
    for i,ch in enumerate(chars):
        if ch==state:
            if st==-1: st=i
        elif st!=-1: seg.append((st,i-1)); st=-1
    if st!=-1: seg.append((st,len(chars)-1)); return seg

def sov_q3(logits,y):
    p=logits.argmax(-1); B=p.size(0); s=0.0
    for i in range(B):
        m=y[i]!=-1; pc=[q3_id_to_char.get(t.item(),'C') for t in p[i][m]]; tc=[q3_id_to_char.get(t.item(),'C') for t in y[i][m]]
        tw=0.0; tr=0.0
        for st in ['H','E','C']:
            ts=get_segments(tc,st); ps=get_segments(pc,st); n=sum(ch==st for ch in tc); tr+=n
            if not ts: continue
            for a,b in ts:
                lo=b-a+1; best=0; bestM=lo; bestL=0
                for c,d in ps:
                    o=max(0,min(b,d)-max(a,c)+1)
                    if o>0:
                        M=max(b,d)-min(a,c)+1; L=d-c+1
                        if o>best: best=o; bestM=M; bestL=L
                if best>0:
                    delta=min(bestM-best,best,lo//2,bestL//2)
                    tw+= (best+delta)/bestM * lo
        if tr>0: s+=tw/tr
    return s/max(1,B)

def train_one_kfold(model_class, name):
    """Train a model using K-fold cross-validation"""
    
    fold_results = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(train_val_indices)):
        print(f"\n{'='*60}")
        print(f"{name} - Fold {fold_idx + 1}/{n_folds}")
        print(f"{'='*60}")
        
        # Create datasets for this fold
        train_fold_indices = [train_val_indices[i] for i in train_idx]
        val_fold_indices = [train_val_indices[i] for i in val_idx]
        
        ds_tr = DS(df.iloc[train_fold_indices]['seq'].tolist(), df.iloc[train_fold_indices]['sst8'].tolist(), df.iloc[train_fold_indices]['sst3'].tolist())
        ds_va = DS(df.iloc[val_fold_indices]['seq'].tolist(), df.iloc[val_fold_indices]['sst8'].tolist(), df.iloc[val_fold_indices]['sst3'].tolist())
        
        train_loader = DataLoader(ds_tr, batch_size=16, shuffle=True, collate_fn=collate, **opt)
        val_loader = DataLoader(ds_va, batch_size=16, shuffle=False, collate_fn=collate, **opt)
        
        print(f"Fold {fold_idx + 1} - Train: {len(ds_tr)}, Val: {len(ds_va)}")
        
        # Initialize model for this fold
        model = model_class().to(device)
        
        # Training setup
        c8 = nn.CrossEntropyLoss(ignore_index=-1)
        c3 = nn.CrossEntropyLoss(ignore_index=-1)
        opt_adam = torch.optim.Adam(model.parameters(), lr=1e-4)
        
        best_val_acc = 0.0
        epochs_no_improve = 0
        patience = 5
        epochs = 50
        
        for ep in range(1, epochs + 1):
            # Training
            model.train()
            tl = ta8 = ta3 = tsv = 0.0
            for x, s8, s3 in tqdm(train_loader, desc=f'{name} Fold {fold_idx+1} Epoch {ep}/{epochs}', leave=False):
                x, s8, s3 = x.to(device), s8.to(device), s3.to(device)
                q8, q3 = model(x)
                l8 = c8(q8.view(-1, 8), s8.view(-1))
                l3 = c3(q3.view(-1, 3), s3.view(-1))
                loss = l8 + 0.5 * l3
                opt_adam.zero_grad()
                loss.backward()
                opt_adam.step()
                tl += loss.item()
                ta8 += acc(q8, s8)
                ta3 += acc(q3, s3)
                tsv += sov_q3(q3, s3)
            
            tl /= len(train_loader)
            ta8 /= len(train_loader)
            ta3 /= len(train_loader)
            tsv /= len(train_loader)
            
            # Validation
            model.eval()
            vl = va8 = va3 = vsv = 0.0
            with torch.no_grad():
                for x, s8, s3 in val_loader:
                    x, s8, s3 = x.to(device), s8.to(device), s3.to(device)
                    q8, q3 = model(x)
                    l8 = c8(q8.view(-1, 8), s8.view(-1))
                    l3 = c3(q3.view(-1, 3), s3.view(-1))
                    loss = l8 + 0.5 * l3
                    vl += loss.item()
                    va8 += acc(q8, s8)
                    va3 += acc(q3, s3)
                    vsv += sov_q3(q3, s3)
            
            vl /= len(val_loader)
            va8 /= len(val_loader)
            va3 /= len(val_loader)
            vsv /= len(val_loader)
            
            print(f'{name} Epoch {ep}: Train Loss={tl:.4f}, Val Loss={vl:.4f}')
            print(f'Train Acc Q8={ta8:.4f}, Val Acc Q8={va8:.4f}')
            print(f'Train Acc Q3={ta3:.4f}, Val Acc Q3={va3:.4f}, Train SOV Q3={tsv:.4f}, Val SOV Q3={vsv:.4f}')
            
            # Early stopping check
            if va8 > best_val_acc:
                best_val_acc = va8
                epochs_no_improve = 0
                ckpt = f'best_{name.lower()}_noemb_fold{fold_idx+1}.pt'
                torch.save(model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(), ckpt)
                print(f'  ✓ Saved best checkpoint for fold {fold_idx+1}')
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f'  Early stopping triggered at epoch {ep}')
                    break
        
        fold_results.append({
            'fold': fold_idx + 1,
            'best_val_acc': best_val_acc
        })
        
        print(f"\nFold {fold_idx + 1} Best Val Acc Q8: {best_val_acc:.4f}")
    
    # Summary
    print(f"\n{'='*60}")
    print(f"{name} K-Fold Cross-Validation Summary")
    print(f"{'='*60}")
    for result in fold_results:
        print(f"Fold {result['fold']}: Val Acc Q8 = {result['best_val_acc']:.4f}")
    
    avg_val_acc = sum([r['best_val_acc'] for r in fold_results]) / n_folds
    print(f"\nAverage Validation Accuracy Q8: {avg_val_acc:.4f}")
    
    # Select best fold
    best_fold = max(fold_results, key=lambda x: x['best_val_acc'])
    print(f"Best Fold: {best_fold['fold']} with Val Acc Q8: {best_fold['best_val_acc']:.4f}")
    
    return best_fold['fold'], fold_results

# Train CNN with K-fold
print("\n" + "="*60)
print("Training CNN with K-Fold Cross-Validation")
print("="*60)
best_cnn_fold, cnn_results = train_one_kfold(ProteinCNN, 'CNN')

# Train BiLSTM with K-fold
print("\n" + "="*60)
print("Training BiLSTM with K-Fold Cross-Validation")
print("="*60)
best_bilstm_fold, bilstm_results = train_one_kfold(ProteinBiLSTM, 'BiLSTM')

# Load best models from best folds
cnn = ProteinCNN().to(device)
cnn.load_state_dict(torch.load(f'best_cnn_noemb_fold{best_cnn_fold}.pt', map_location=device))
cnn.eval()

bilstm = ProteinBiLSTM().to(device)
bilstm.load_state_dict(torch.load(f'best_bilstm_noemb_fold{best_bilstm_fold}.pt', map_location=device))
bilstm.eval()

print(f"\nLoaded best CNN from fold {best_cnn_fold}")
print(f"Loaded best BiLSTM from fold {best_bilstm_fold}")

'trained'


## 5. Ensemble Evaluation
Combines predictions from the best CNN and BiLSTM models (from K-fold cross-validation) using optimized alpha weights tuned on validation set, and evaluates on test set.

In [ ]:
@torch.no_grad()
def collect(loader):
    q8c,q3c,q8b,q3b,s8s,s3s=[],[],[],[],[],[]
    for x,s8,s3 in loader:
        x,s8,s3=x.to(device),s8.to(device),s3.to(device)
        a8,a3=cnn(x); b8,b3=bilstm(x)
        q8c.append(a8.cpu()); q3c.append(a3.cpu()); q8b.append(b8.cpu()); q3b.append(b3.cpu()); s8s.append(s8.cpu()); s3s.append(s3.cpu())
    import torch as T
    return T.cat(q8c),T.cat(q3c),T.cat(q8b),T.cat(q3b),T.cat(s8s),T.cat(s3s)
def masked_acc(logits, labels):
    p=logits.argmax(-1); m=labels!=-1; import torch as T;
    return (p[m]==labels[m]).float().mean().item() if m.any() else 0.0
v_q8c,v_q3c,v_q8b,v_q3b,v_s8,v_s3=collect(val_loader)
import torch as T
alphas=T.linspace(0,1,21); best_a8=0.5; best_v8=-1; best_a3=0.5; best_v3=-1
for a in alphas:
    acc8=masked_acc(a*v_q8c+(1-a)*v_q8b, v_s8); acc3=masked_acc(a*v_q3c+(1-a)*v_q3b, v_s3)
    if acc8>best_v8: best_v8, best_a8 = acc8, float(a)
    if acc3>best_v3: best_v3, best_a3 = acc3, float(a)
print(f'Best alpha Q8: {best_a8:.2f} | Val Acc: {best_v8:.4f}')
print(f'Best alpha Q3: {best_a3:.2f} | Val Acc: {best_v3:.4f}')
@torch.no_grad()
def evaluate(loader,a8,a3):
    tot8=tot3=0; cor8=cor3=0
    for x,s8,s3 in loader:
        x,s8,s3=x.to(device),s8.to(device),s3.to(device)
        c8,c3=cnn(x); b8,b3=bilstm(x)
        q8=a8*c8+(1-a8)*b8; q3=a3*c3+(1-a3)*b3
        p8=q8.argmax(-1); p3=q3.argmax(-1); m8=s8!=-1; m3=s3!=-1
        cor8+=(p8[m8]==s8[m8]).sum().item(); tot8+=m8.sum().item(); cor3+=(p3[m3]==s3[m3]).sum().item(); tot3+=m3.sum().item()
    return cor8/max(1,tot8), cor3/max(1,tot3)
t8,t3=evaluate(test_loader,best_a8,best_a3)
print(f'Test Accuracy Q8: {t8:.4f}')
print(f'Test Accuracy Q3: {t3:.4f}')
